# Activation Functions in PyTorch

For each activation function we:
1. **Implement from scratch** using basic tensor ops
2. **Verify numerically** against `torch.nn`
3. **Inspect the gradient** via `.backward()`
4. **Visualize** the function and its derivative over $[-5, 5]$

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print(f'PyTorch version: {torch.__version__}')

def check(name, a, b, atol=1e-5):
    ok = torch.allclose(a.float(), b.float(), atol=atol)
    status = '✓' if ok else '✗'
    print(f'{status} {name}: max_diff={torch.abs(a.float()-b.float()).max().item():.2e}')

x_vis = torch.linspace(-5, 5, 500)

## Section 1 — The ReLU Family

ReLU, LeakyReLU, PReLU, RReLU, ReLU6

In [ ]:
x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])

# ReLU
def relu_scratch(x):       return torch.clamp(x, min=0)
check('ReLU',       relu_scratch(x), nn.ReLU()(x))

# LeakyReLU (alpha=0.01)
def leaky_relu_scratch(x, alpha=0.01): return torch.where(x >= 0, x, alpha * x)
check('LeakyReLU',  leaky_relu_scratch(x), nn.LeakyReLU(0.01)(x))

# ReLU6
def relu6_scratch(x):      return torch.clamp(x, min=0, max=6)
check('ReLU6',      relu6_scratch(x), nn.ReLU6()(x))

# PReLU (fixed alpha=0.25 for test)
alpha_prelu = 0.25
def prelu_scratch(x, a):   return torch.where(x >= 0, x, a * x)
prelu_mod = nn.PReLU()
prelu_mod.weight.data.fill_(alpha_prelu)
check('PReLU',      prelu_scratch(x, alpha_prelu), prelu_mod(x))

# RReLU — test in eval mode (uses fixed midpoint)
lower, upper = 1/8, 1/3
a_fixed = (lower + upper) / 2
rrelu_mod = nn.RReLU(lower, upper)
rrelu_mod.eval()
check('RReLU (eval)', prelu_scratch(x, a_fixed), rrelu_mod(x))

In [ ]:
# Visualise ReLU family
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(x_vis, relu_scratch(x_vis),              label='ReLU',      linewidth=2)
ax.plot(x_vis, leaky_relu_scratch(x_vis, 0.1),   label='Leaky(0.1)',linewidth=2)
ax.plot(x_vis, relu6_scratch(x_vis),             label='ReLU6',     linewidth=2)
ax.set_title('ReLU Family'); ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.set_ylim(-1, 6); ax.grid(True, alpha=0.3)

# Gradient comparison
ax = axes[1]
ax.plot(x_vis, (x_vis > 0).float(),             label='ReLU gradient', linewidth=2)
ax.plot(x_vis, torch.where(x_vis >= 0, torch.ones_like(x_vis), torch.full_like(x_vis, 0.1)),
        label='Leaky gradient', linewidth=2)
ax.plot(x_vis, ((x_vis > 0) & (x_vis < 6)).float(), label='ReLU6 gradient', linewidth=2)
ax.set_title('Gradients'); ax.set_xlabel('x'); ax.set_ylabel("f'(x)")
ax.legend(); ax.set_ylim(-0.1, 1.2); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Section 2 — Saturating Activations

Sigmoid, Tanh, Hardsigmoid, Hardtanh, Softsign, LogSigmoid

In [ ]:
x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])

# Sigmoid
def sigmoid_scratch(x):     return 1 / (1 + torch.exp(-x))
check('Sigmoid',     sigmoid_scratch(x), torch.sigmoid(x))

# Sigmoid derivative: σ(x)·(1-σ(x)), max = 0.25
x2 = x.clone().requires_grad_(True)
torch.sigmoid(x2).sum().backward()
analytical_sig_grad = torch.sigmoid(x) * (1 - torch.sigmoid(x))
check('Sigmoid grad', x2.grad, analytical_sig_grad)
print(f'  Max sigmoid gradient: {x2.grad.max().item():.4f} (should be ≤ 0.25)')

# Tanh
def tanh_scratch(x): return (torch.exp(x) - torch.exp(-x)) / (torch.exp(x) + torch.exp(-x))
check('Tanh',        tanh_scratch(x), torch.tanh(x))

# Tanh derivative: 1 - tanh²(x), max = 1
x3 = x.clone().requires_grad_(True)
torch.tanh(x3).sum().backward()
analytical_tanh_grad = 1 - torch.tanh(x)**2
check('Tanh grad',   x3.grad, analytical_tanh_grad)

# Hardsigmoid: max(0, min(1, (x+3)/6))
def hardsig_scratch(x): return torch.clamp((x + 3) / 6, 0, 1)
check('Hardsigmoid', hardsig_scratch(x), nn.Hardsigmoid()(x))

# Hardtanh
def hardtanh_scratch(x, lo=-1, hi=1): return torch.clamp(x, lo, hi)
check('Hardtanh',    hardtanh_scratch(x), nn.Hardtanh()(x))

# Softsign
def softsign_scratch(x): return x / (1 + torch.abs(x))
check('Softsign',    softsign_scratch(x), nn.Softsign()(x))

# LogSigmoid (numerically stable)
def logsig_scratch(x): return torch.where(x >= 0, -torch.log(1+torch.exp(-x)), x - torch.log(1+torch.exp(x)))
check('LogSigmoid',  logsig_scratch(x), nn.LogSigmoid()(x))

In [ ]:
# Vanishing gradient demo: compare gradients at depth 20
with torch.no_grad():
    sig_max_grad = 0.25
    tanh_max_grad = 1.0
    depths = list(range(1, 25))
    sig_grads  = [sig_max_grad**d for d in depths]
    tanh_grads = [tanh_max_grad**d for d in depths]

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(depths, sig_grads,  'o-', label='Sigmoid (max 0.25 per layer)', linewidth=2)
ax.semilogy(depths, tanh_grads, 's-', label='Tanh (max 1.0 per layer)', linewidth=2)
ax.axhline(1e-7, color='red', linestyle='--', label='Numerical zero threshold')
ax.set_xlabel('Network depth'); ax.set_ylabel('Gradient magnitude (log scale)')
ax.set_title('Vanishing Gradient: Upper Bound on Backprop Signal')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Section 3 — Smooth Modern Activations

GELU, SiLU/Swish, Mish, ELU, CELU, SELU

In [ ]:
x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])

# GELU (tanh approximation)
def gelu_scratch(x):
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2/math.pi) * (x + 0.044715 * x**3)))
check('GELU (tanh approx)', gelu_scratch(x), nn.GELU()(x))
check('GELU (exact)',       F.gelu(x, approximate='none'), nn.GELU(approximate='none')(x))

# SiLU / Swish: x * σ(x)
def silu_scratch(x): return x * torch.sigmoid(x)
check('SiLU',        silu_scratch(x), nn.SiLU()(x))

# Mish: x * tanh(softplus(x))
def mish_scratch(x): return x * torch.tanh(F.softplus(x))
check('Mish',        mish_scratch(x), nn.Mish()(x))

# ELU (alpha=1.0)
def elu_scratch(x, alpha=1.0): return torch.where(x > 0, x, alpha * (torch.exp(x) - 1))
check('ELU',         elu_scratch(x), nn.ELU()(x))

# CELU: max(0,x) + min(0, alpha*(exp(x/alpha)-1))
def celu_scratch(x, alpha=1.0):
    return torch.clamp(x, min=0) + torch.clamp(alpha*(torch.exp(x/alpha)-1), max=0)
check('CELU',        celu_scratch(x), nn.CELU()(x))

# SELU: lambda * ELU(x, alpha) — constants are fixed
SELU_LAMBDA = 1.0507009873554804934193349852946
SELU_ALPHA  = 1.6732632423543772848170429916717
def selu_scratch(x):
    return SELU_LAMBDA * torch.where(x > 0, x, SELU_ALPHA * (torch.exp(x) - 1))
check('SELU',        selu_scratch(x), nn.SELU()(x))

In [ ]:
# Compare smooth activations visually
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
with torch.no_grad():
    ax.plot(x_vis, F.relu(x_vis),    label='ReLU',      linewidth=1.5, linestyle='--')
    ax.plot(x_vis, gelu_scratch(x_vis), label='GELU',   linewidth=2)
    ax.plot(x_vis, silu_scratch(x_vis), label='SiLU',   linewidth=2)
    ax.plot(x_vis, mish_scratch(x_vis), label='Mish',   linewidth=2)
    ax.plot(x_vis, elu_scratch(x_vis),  label='ELU',    linewidth=2)
ax.set_title('Smooth Activations'); ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.set_ylim(-2, 5); ax.grid(True, alpha=0.3)

ax = axes[1]
x_grad = x_vis.clone().requires_grad_(False)
def numerical_grad(fn, x, eps=1e-4):
    return (fn(x + eps) - fn(x - eps)) / (2 * eps)

with torch.no_grad():
    ax.plot(x_vis, (x_vis > 0).float(),                 label="ReLU'",  linewidth=1.5, linestyle='--')
    ax.plot(x_vis, numerical_grad(gelu_scratch, x_vis), label="GELU'",  linewidth=2)
    ax.plot(x_vis, numerical_grad(silu_scratch, x_vis), label="SiLU'",  linewidth=2)
    ax.plot(x_vis, numerical_grad(mish_scratch, x_vis), label="Mish'",  linewidth=2)
    ax.plot(x_vis, numerical_grad(elu_scratch, x_vis),  label="ELU'",   linewidth=2)
ax.set_title('Derivatives'); ax.set_xlabel('x'); ax.set_ylabel("f'(x)")
ax.legend(); ax.set_ylim(-0.3, 1.4); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Section 4 — Gating & Normalization

GLU, Hardswish, Softmax, LogSoftmax, Softmax2d, Softmin

In [ ]:
# GLU: splits input in half, applies sigmoid to second half
x_glu = torch.randn(4, 8)  # will be split into two (4,4) tensors
def glu_scratch(x, dim=-1):
    a, b = x.chunk(2, dim=dim)
    return a * torch.sigmoid(b)
check('GLU', glu_scratch(x_glu), nn.GLU()(x_glu))
print(f'  GLU input shape: {x_glu.shape} → output shape: {glu_scratch(x_glu).shape}  (halved)')

# Hardswish: x * ReLU6(x+3) / 6
x = torch.tensor([-4.0, -1.0, 0.0, 1.0, 4.0])
def hardswish_scratch(x): return x * torch.clamp(x + 3, 0, 6) / 6
check('Hardswish', hardswish_scratch(x), nn.Hardswish()(x))

# Softmax
logits = torch.tensor([[1.0, 2.0, 0.5], [0.0, 3.0, 1.0]])
def softmax_scratch(x, dim=-1):
    x_max = x.max(dim=dim, keepdim=True).values  # stable: subtract max
    ex = torch.exp(x - x_max)
    return ex / ex.sum(dim=dim, keepdim=True)
check('Softmax', softmax_scratch(logits), F.softmax(logits, dim=-1))
print(f'  Softmax sums: {softmax_scratch(logits).sum(-1).tolist()}  (should be [1.0, 1.0])')

# LogSoftmax: x_i - log(sum(exp(x_j)))
def log_softmax_scratch(x, dim=-1):
    x_max = x.max(dim=dim, keepdim=True).values
    return (x - x_max) - torch.log(torch.exp(x - x_max).sum(dim=dim, keepdim=True))
check('LogSoftmax', log_softmax_scratch(logits), F.log_softmax(logits, dim=-1))

# Softmax2d: softmax over channel dim (dim=1) for (N,C,H,W) tensors
feat_map = torch.randn(2, 4, 3, 3)  # (N,C,H,W)
sm2d = nn.Softmax2d()(feat_map)
manual_sm2d = F.softmax(feat_map, dim=1)
check('Softmax2d', sm2d, manual_sm2d)
print(f'  Channel sum at pixel (0,0,0): {sm2d[0, :, 0, 0].sum().item():.6f}  (should be 1.0)')

# Softmin: softmax(-x)
x_vec = torch.tensor([[1.0, 2.0, 3.0]])
check('Softmin', F.softmax(-x_vec, dim=-1), F.softmin(x_vec, dim=-1))
print(f'  Softmin: higher prob for smaller value → {F.softmin(x_vec, dim=-1).tolist()}')

## Section 5 — Shrinkage & Threshold Functions

In [ ]:
x = torch.tensor([-2.0, -0.3, 0.0, 0.3, 2.0])
lambd = 0.5

# Hardshrink: x if |x|>λ, else 0
def hardshrink_scratch(x, lam=0.5):
    return torch.where(torch.abs(x) > lam, x, torch.zeros_like(x))
check('Hardshrink', hardshrink_scratch(x), nn.Hardshrink(lambd)(x))

# Softshrink: sign(x)*max(0, |x|-λ)
def softshrink_scratch(x, lam=0.5):
    return torch.sign(x) * torch.clamp(torch.abs(x) - lam, min=0)
check('Softshrink', softshrink_scratch(x), nn.Softshrink(lambd)(x))

# Tanhshrink: x - tanh(x)
def tanhshrink_scratch(x): return x - torch.tanh(x)
check('Tanhshrink', tanhshrink_scratch(x), nn.Tanhshrink()(x))

# Threshold: x if x>t else value
def threshold_scratch(x, t=0.1, v=0.0):
    return torch.where(x > t, x, torch.full_like(x, v))
check('Threshold', threshold_scratch(x), nn.Threshold(0.1, 0.0)(x))

# Softplus: (1/β)*log(1 + exp(βx))
def softplus_scratch(x, beta=1, threshold=20):
    return torch.where(beta * x > threshold, x,
                       (1/beta) * torch.log(1 + torch.exp(beta * x)))
check('Softplus', softplus_scratch(x), nn.Softplus()(x))

# Softplus gradient = sigmoid
x_sp = x.clone().requires_grad_(True)
nn.Softplus()(x_sp).sum().backward()
check('Softplus grad = sigmoid', x_sp.grad, torch.sigmoid(x))
print('Softplus gradient = sigmoid ✓')

In [ ]:
# Compare shrinkage functions
x_plot = torch.linspace(-3, 3, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(x_plot, hardshrink_scratch(x_plot, 0.5),  label='Hardshrink(λ=0.5)', linewidth=2)
ax.plot(x_plot, softshrink_scratch(x_plot, 0.5),  label='Softshrink(λ=0.5)', linewidth=2)
ax.plot(x_plot, tanhshrink_scratch(x_plot),       label='Tanhshrink',        linewidth=2)
ax.axvline(-0.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline( 0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Shrinkage Functions'); ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_plot, softplus_scratch(x_plot), label='Softplus (β=1)', linewidth=2)
ax.plot(x_plot, F.relu(x_plot),           label='ReLU (β→∞)',    linewidth=2, linestyle='--')
ax.set_title('Softplus vs ReLU'); ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.set_ylim(-0.2, 4); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Section 6 — Dying ReLU Experiment

We track the fraction of dead neurons (always outputting 0) over training with ReLU vs LeakyReLU vs ELU.

In [ ]:
def count_dead_neurons(model, X, threshold=1e-6):
    """Fraction of hidden neurons that always output ~0 on dataset X."""
    model.eval()
    with torch.no_grad():
        # Collect activations from all hidden layers
        activations = []
        hook_handles = []
        for layer in model.modules():
            if isinstance(layer, nn.Linear):
                def hook(m, inp, out, acts=activations): acts.append(out)
                hook_handles.append(layer.register_forward_hook(hook))
        
        _ = model(X)
        for h in hook_handles: h.remove()
        
        # A neuron is dead if max(|activation|) across all samples < threshold
        dead_count = total = 0
        for act in activations[:-1]:  # exclude output layer
            max_per_neuron = act.abs().max(dim=0).values
            dead_count += (max_per_neuron < threshold).sum().item()
            total += act.shape[1]
        return dead_count / total if total > 0 else 0.0

def build_mlp(activation):
    return nn.Sequential(
        nn.Linear(20, 64), activation,
        nn.Linear(64, 64), activation,
        nn.Linear(64, 1)
    )

# Use a large learning rate + negative bias init to provoke dying ReLU
def init_dead_biases(model):
    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            nn.init.constant_(layer.bias, -2.0)  # large negative bias → dying ReLU

torch.manual_seed(0)
X_data = torch.randn(512, 20)
y_data = X_data[:, :3].sum(1, keepdim=True)

models = {
    'ReLU':       build_mlp(nn.ReLU()),
    'LeakyReLU':  build_mlp(nn.LeakyReLU(0.01)),
    'ELU':        build_mlp(nn.ELU()),
}

# Initialize with negative biases to trigger dying ReLU
for m in models.values(): init_dead_biases(m)

dead_history = {name: [] for name in models}
n_epochs = 100

for name, model in models.items():
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    for ep in range(n_epochs):
        model.train()
        opt.zero_grad()
        loss = F.mse_loss(model(X_data), y_data)
        loss.backward()
        opt.step()
        dead_history[name].append(count_dead_neurons(model, X_data))
    print(f'{name}: final dead neuron fraction = {dead_history[name][-1]:.2%}')

plt.figure(figsize=(9, 4))
for name, hist in dead_history.items():
    plt.plot(hist, label=name, linewidth=2)
plt.xlabel('Training step'); plt.ylabel('Dead neuron fraction')
plt.title('Dying Neuron Experiment (large negative bias init, SGD lr=0.1)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Section 7 — SwiGLU FFN

SwiGLU = SiLU(xW₁) ⊗ (xW₃), followed by projection W₂. Used in LLaMA, Mistral, PaLM.

In [ ]:
class SwiGLUFFN(nn.Module):
    """SwiGLU feedforward block as in LLaMA."""
    def __init__(self, d_model, hidden_dim=None):
        super().__init__()
        # Scale hidden dim to ~8/3 of d_model (maintains FLOPs vs 4x ReLU FFN)
        if hidden_dim is None:
            hidden_dim = int(8 * d_model / 3 / 64 + 1) * 64  # round to 64
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)  # gate projection
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)  # output projection
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)  # up projection
        self.silu = nn.SiLU()
    
    def forward(self, x):
        return self.w2(self.silu(self.w1(x)) * self.w3(x))

class ReLUFFN(nn.Module):
    def __init__(self, d_model, hidden_dim=None):
        super().__init__()
        if hidden_dim is None: hidden_dim = d_model * 4
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, d_model)
        )
    def forward(self, x): return self.net(x)

d_model = 128
swiglu = SwiGLUFFN(d_model)
relu_ffn = ReLUFFN(d_model)
x_ffn = torch.randn(4, 16, d_model)  # (batch, seq, d_model)
print(f'SwiGLU output shape: {swiglu(x_ffn).shape}')
print(f'ReLU FFN output shape: {relu_ffn(x_ffn).shape}')
print(f'SwiGLU params: {sum(p.numel() for p in swiglu.parameters()):,}')
print(f'ReLU FFN params: {sum(p.numel() for p in relu_ffn.parameters()):,}')

In [ ]:
# Compare convergence: SwiGLU vs ReLU FFN on a simple regression task
class SimpleModel(nn.Module):
    def __init__(self, ffn):
        super().__init__()
        self.input_proj = nn.Linear(10, d_model)
        self.ffn = ffn
        self.output_proj = nn.Linear(d_model, 1)
    def forward(self, x):
        x = F.relu(self.input_proj(x))[:, None, :].expand(-1, 4, -1)  # fake seq
        x = self.ffn(x)
        return self.output_proj(x[:, 0, :])

torch.manual_seed(1)
X_syn = torch.randn(2000, 10)
y_syn = (X_syn[:, 0]**2 + X_syn[:, 1] * X_syn[:, 2] - X_syn[:, 3]).unsqueeze(1)

histories = {}
for name, ffn in [('SwiGLU', SwiGLUFFN(d_model)), ('ReLU FFN', ReLUFFN(d_model))]:
    model = SimpleModel(ffn)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for ep in range(50):
        model.train()
        opt.zero_grad()
        loss = F.mse_loss(model(X_syn), y_syn)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    histories[name] = losses
    print(f'{name}: final loss = {losses[-1]:.4f}')

plt.figure(figsize=(8, 4))
for name, hist in histories.items():
    plt.plot(hist, label=name, linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('SwiGLU vs ReLU FFN Convergence')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Section 8 — Compute Benchmarks

Compare wall-clock time per activation on a (1000, 512) tensor.

In [ ]:
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x_bench = torch.randn(1000, 512, device=device)

activations_to_bench = {
    'ReLU':       nn.ReLU(),
    'ReLU6':      nn.ReLU6(),
    'Hardsigmoid': nn.Hardsigmoid(),
    'Hardswish':  nn.Hardswish(),
    'Sigmoid':    nn.Sigmoid(),
    'Tanh':       nn.Tanh(),
    'SiLU':       nn.SiLU(),
    'GELU':       nn.GELU(),
    'ELU':        nn.ELU(),
    'SELU':       nn.SELU(),
    'Mish':       nn.Mish(),
    'Softplus':   nn.Softplus(),
}

timings = {}
N_WARMUP, N_RUNS = 20, 200

for name, act in activations_to_bench.items():
    act = act.to(device)
    for _ in range(N_WARMUP):
        _ = act(x_bench)
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = act(x_bench)
    if device.type == 'cuda': torch.cuda.synchronize()
    timings[name] = (time.perf_counter() - t0) / N_RUNS * 1000  # ms

# Sort and display
sorted_timings = sorted(timings.items(), key=lambda kv: kv[1])
print(f'\nActivation benchmarks on {device} — {x_bench.shape} tensor:')
print(f'{"Activation":<16} {"Time (ms)":<12} {"Relative"}')
print('-' * 40)
baseline = sorted_timings[0][1]
for name, t in sorted_timings:
    bar = '█' * int(t / baseline)
    print(f'{name:<16} {t:<12.4f} {bar}')

## Section 9 — End-to-End: CIFAR-10 Activation Comparison

Train a 4-layer MLP with 5 different activations and compare validation accuracy.

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_set = torchvision.datasets.CIFAR10(root='/tmp/cifar', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='/tmp/cifar', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=256, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)
print(f'CIFAR-10: {len(train_set)} train, {len(test_set)} test')

In [ ]:
class MLP(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3072, 512), activation,
            nn.Linear(512, 512),  activation,
            nn.Linear(512, 256),  activation,
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.net(x.view(x.size(0), -1))

activation_configs = {
    'ReLU':  nn.ReLU(),
    'SiLU':  nn.SiLU(),
    'GELU':  nn.GELU(),
    'Mish':  nn.Mish(),
    'ELU':   nn.ELU(),
}

val_acc_histories = {}

for act_name, act_fn in activation_configs.items():
    torch.manual_seed(42)
    model = MLP(act_fn).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    val_accs = []
    
    for epoch in range(10):
        # Train
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(imgs), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()
        
        # Evaluate
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total
        val_accs.append(val_acc)
    
    val_acc_histories[act_name] = val_accs
    print(f'{act_name:<10}: final val acc = {val_accs[-1]:.4f}')

plt.figure(figsize=(9, 5))
for name, hist in val_acc_histories.items():
    plt.plot(range(1, 11), hist, 'o-', label=name, linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy')
plt.title('CIFAR-10 MLP: Activation Function Comparison')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()